# Exercise 2.1: MCP Concepts — The Universal Adapter for AI Tools

**Module:** 2 — Model Context Protocol
**Level:** Conceptual

In this notebook, you'll learn how MCP (Model Context Protocol) works — the open standard that lets AI models discover and use tools.

**What you'll do:**
1. Understand what MCP is and why it exists
2. Explore FastMCP — the high-level API for building MCP servers
3. Define tools that AI clients can discover and call
4. Define resources that AI clients can read
5. See the JSON-RPC format (what messages look like on the wire)
6. Test tool functions directly as regular Python

> **Note:** MCP servers require a persistent process (stdio/SSE), which doesn't work in Colab.
> So we'll explore the **concepts and code patterns** without starting a server.

## 1. Setup

Install the MCP SDK. This gives us the `FastMCP` class and all the decorators.

In [ ]:
# Install the MCP Python SDK
# This package provides FastMCP, decorators for tools/resources, and the protocol layer
!pip install mcp -q

## 2. What is MCP?

**Problem:** Every AI tool integration is custom. OpenAI has function calling, Anthropic has tool use, LangChain has its own format. If you build a tool, you have to integrate it separately with each AI system.

**Solution:** MCP (Model Context Protocol) is an **open standard** — like USB for AI tools.

```
Before MCP:                          With MCP:
┌─────────┐    custom    ┌──────┐    ┌─────────┐              ┌──────┐
│ Claude  │──────────────│ Tool │    │ Claude  │───┐          │ Tool │
└─────────┘              └──────┘    └─────────┘   │   MCP    └──┬───┘
┌─────────┐    custom    ┌──────┐    ┌─────────┐   ├──────────┤
│ ChatGPT │──────────────│ Tool │    │ ChatGPT │───┘          │ Tool │
└─────────┘              └──────┘    └─────────┘              └──────┘
```

**Key concepts:**
- **MCP Server:** Exposes tools and resources (you build this)
- **MCP Client:** The AI application that discovers and calls tools (Claude, Cursor, etc.)
- **Transport:** How they communicate (stdio, SSE, HTTP)

## 3. FastMCP — The High-Level API

**FastMCP** is the high-level API for MCP servers, like **Express.js for web servers** or **Flask for Python APIs**.

Instead of dealing with raw JSON-RPC messages, you just write normal Python functions and decorate them.

In [ ]:
from mcp.server.fastmcp import FastMCP

# Create an MCP server instance.
# Think of this as creating a Flask app — it's the container for all your tools.
# The name "Amadeus Travel Tools" is what AI clients see when they connect.
mcp = FastMCP("Amadeus Travel Tools")

print(f"Server name: {mcp.name}")
print(f"Type: {type(mcp)}")
print("\nServer created! Now let's add tools and resources to it.")

## 4. Defining Tools with `@mcp.tool()`

A **tool** is an action the AI can **perform** — like checking flight status, booking a ticket, or calculating a price.

The `@mcp.tool()` decorator **registers** this function so AI clients can **discover and call it**. The function's docstring becomes the tool's description, and the parameters become the tool's input schema — automatically.

In [ ]:
@mcp.tool()
def check_flight_status(flight_number: str, date: str) -> str:
    """Check the current status of a flight.

    Args:
        flight_number: IATA flight number (e.g., 'TK1234')
        date: Date in YYYY-MM-DD format
    """
    # In production, this would call the Amadeus API.
    # For now, we return mock data to show the pattern.
    mock_data = {
        "TK1234": {"status": "On Time", "departure": "09:30", "gate": "A12"},
        "TK5678": {"status": "Delayed", "departure": "14:45", "gate": "B7"},
    }

    # Look up the flight in our mock database
    flight = mock_data.get(flight_number)

    if flight:
        return f"Flight {flight_number} on {date}: {flight['status']}, departs {flight['departure']} from gate {flight['gate']}"
    else:
        return f"Flight {flight_number} not found"


# The decorator does two things:
# 1. Registers the function with the MCP server
# 2. Extracts the schema from type hints and docstring
print("Tool registered: check_flight_status")
print("AI clients will see the function name, description, and parameter types.")

In [ ]:
# Let's add a second tool — a price calculator
@mcp.tool()
def calculate_fare(origin: str, destination: str, cabin_class: str = "economy") -> str:
    """Calculate estimated fare between two airports.

    Args:
        origin: Origin airport IATA code (e.g., 'IST')
        destination: Destination airport IATA code (e.g., 'CDG')
        cabin_class: Cabin class — economy, business, or first
    """
    # Mock pricing logic — in reality this would call a pricing API
    base_prices = {
        ("IST", "CDG"): 250,
        ("IST", "JFK"): 650,
        ("CDG", "NRT"): 800,
    }

    # Multipliers for cabin class upgrades
    multipliers = {"economy": 1.0, "business": 2.5, "first": 4.0}

    base = base_prices.get((origin, destination), 500)  # Default 500 if route unknown
    multiplier = multipliers.get(cabin_class, 1.0)
    fare = base * multiplier

    return f"{origin} → {destination} ({cabin_class}): ${fare:.0f} estimated"


print("Tool registered: calculate_fare")
print("Notice: cabin_class has a default value — MCP marks it as optional.")

## 5. Defining Resources with `@mcp.resource()`

A **resource** is data the AI can **read** — like a database record, a config file, or a policy document.

Think of the difference this way:
- **Tools** = actions (verbs): check status, calculate price, book ticket
- **Resources** = data (nouns): airline policies, airport info, fare rules

Resources use **URI patterns** (like `policy://cancellation`) so the AI can request specific data.

In [ ]:
@mcp.resource("policy://cancellation")
def get_cancellation_policy() -> str:
    """Returns the airline's cancellation policy."""
    # The URI "policy://cancellation" is how the AI asks for this specific resource.
    # When an AI client reads this resource, it gets the policy text.
    return """CANCELLATION POLICY:
    - Free cancellation within 24 hours of booking
    - 25% fee for cancellations 7+ days before departure
    - 50% fee for cancellations 2-7 days before departure
    - No refund for cancellations less than 48 hours before departure
    - Business/First class: free changes up to 4 hours before departure"""


@mcp.resource("airport://{code}")
def get_airport_info(code: str) -> str:
    """Returns information about an airport by its IATA code."""
    # The {code} in the URI is a parameter — "airport://IST" fills code="IST".
    # This is like a URL parameter in a REST API.
    airports = {
        "IST": {"name": "Istanbul Airport", "city": "Istanbul", "terminals": 2, "timezone": "Europe/Istanbul"},
        "CDG": {"name": "Charles de Gaulle", "city": "Paris", "terminals": 3, "timezone": "Europe/Paris"},
        "JFK": {"name": "John F. Kennedy", "city": "New York", "terminals": 6, "timezone": "America/New_York"},
    }

    info = airports.get(code.upper())
    if info:
        return f"{info['name']} ({code.upper()}) — {info['city']}, {info['terminals']} terminals, {info['timezone']}"
    else:
        return f"Airport {code.upper()} not found in database"


print("Resources registered:")
print("  - policy://cancellation (static — always returns the same document)")
print("  - airport://{code}      (dynamic — different data for each airport code)")

## 6. Testing Tools and Resources Directly

Since we can't start an MCP server in Colab, let's test our functions the old-fashioned way — by calling them as regular Python functions.

This is actually a valid development workflow: **define your tool logic first, test it, then wrap it with MCP.**

In [ ]:
# Test the flight status tool
# Even though it's decorated with @mcp.tool(), it's still a normal Python function.
print("=== Flight Status Tool ===")
print(check_flight_status("TK1234", "2026-04-15"))
print(check_flight_status("TK5678", "2026-04-15"))
print(check_flight_status("XX9999", "2026-04-15"))  # Unknown flight

In [ ]:
# Test the fare calculator
print("=== Fare Calculator Tool ===")
print(calculate_fare("IST", "CDG"))                  # Default: economy
print(calculate_fare("IST", "JFK", "business"))      # Business class
print(calculate_fare("CDG", "NRT", "first"))         # First class
print(calculate_fare("LAX", "SFO"))                  # Unknown route, uses default price

In [ ]:
# Test the resources
print("=== Cancellation Policy Resource ===")
print(get_cancellation_policy())
print()
print("=== Airport Info Resource ===")
print(get_airport_info("IST"))
print(get_airport_info("CDG"))
print(get_airport_info("XYZ"))  # Unknown airport

## 7. What Happens on the Wire — JSON-RPC

When an AI client talks to an MCP server, they exchange **JSON-RPC** messages. This is the protocol format — like HTTP for web APIs, but simpler.

You don't write these messages yourself (FastMCP handles it), but understanding them helps you debug and understand what's happening.

Here's what the conversation looks like:

In [ ]:
import json

# Step 1: Client discovers available tools
# The AI client sends this request to find out what tools the server offers.
discover_request = {
    "jsonrpc": "2.0",          # JSON-RPC version (always 2.0)
    "id": 1,                    # Request ID (for matching responses)
    "method": "tools/list",     # What we're asking: "list your tools"
    "params": {}                # No parameters needed for listing
}

print("=== Step 1: Client asks 'What tools do you have?' ===")
print(json.dumps(discover_request, indent=2))

In [ ]:
# Step 2: Server responds with the tool list
# This is what the server sends back — the schema of each tool.
discover_response = {
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "tools": [
            {
                "name": "check_flight_status",
                "description": "Check the current status of a flight.",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "flight_number": {"type": "string", "description": "IATA flight number (e.g., 'TK1234')"},
                        "date": {"type": "string", "description": "Date in YYYY-MM-DD format"}
                    },
                    "required": ["flight_number", "date"]
                }
            },
            {
                "name": "calculate_fare",
                "description": "Calculate estimated fare between two airports.",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "origin": {"type": "string", "description": "Origin airport IATA code"},
                        "destination": {"type": "string", "description": "Destination airport IATA code"},
                        "cabin_class": {"type": "string", "description": "economy, business, or first"}
                    },
                    "required": ["origin", "destination"]
                }
            }
        ]
    }
}

print("=== Step 2: Server responds with tool schemas ===")
print(json.dumps(discover_response, indent=2))
print("\nThe AI client now knows what tools exist, what they do, and what parameters they need.")

In [ ]:
# Step 3: Client calls a tool
# The AI decides to check a flight status, so it sends this:
call_request = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",          # "I want to call a tool"
    "params": {
        "name": "check_flight_status",  # Which tool to call
        "arguments": {                   # The inputs for the tool
            "flight_number": "TK1234",
            "date": "2026-04-15"
        }
    }
}

print("=== Step 3: Client calls check_flight_status ===")
print(json.dumps(call_request, indent=2))

# Step 4: Server executes the function and returns the result
call_response = {
    "jsonrpc": "2.0",
    "id": 2,
    "result": {
        "content": [
            {
                "type": "text",
                "text": "Flight TK1234 on 2026-04-15: On Time, departs 09:30 from gate A12"
            }
        ]
    }
}

print("\n=== Step 4: Server returns the result ===")
print(json.dumps(call_response, indent=2))
print("\nThe AI gets this result and uses it to answer the user's question.")

## 8. The Full Picture

Here's how all the pieces fit together in a real MCP deployment:

```
┌──────────────────┐     JSON-RPC      ┌──────────────────────┐
│    AI Client     │ ←──────────────→  │     MCP Server       │
│  (Claude, etc.)  │                   │  (Your FastMCP app)  │
│                  │  1. tools/list     │                      │
│  "What can I do?"│ ────────────────→ │  Returns tool schemas│
│                  │                   │                      │
│  User asks:      │  2. tools/call    │  @mcp.tool()         │
│  "Is TK1234      │ ────────────────→ │  check_flight_status │
│   on time?"      │                   │  → calls Amadeus API │
│                  │  3. result        │                      │
│  "Your flight is"│ ←──────────────── │  Returns flight data │
│   on time..."    │                   │                      │
└──────────────────┘                   └──────────────────────┘
```

**In production**, you'd run the MCP server as a separate process and connect to it via stdio or SSE. But the tool/resource logic you write is exactly the same as what we wrote above.

## 9. How You'd Run This Server (Outside Colab)

In a real environment, you'd add one line to start the server:

In [ ]:
# DO NOT RUN THIS IN COLAB — it requires a persistent process.
# This is just to show the pattern.

# In a file called "server.py", you'd write:
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Amadeus Travel Tools")

@mcp.tool()
def check_flight_status(flight_number: str, date: str) -> str:
    """Check the current status of a flight."""
    # ... your logic here ...
    return f"Flight {flight_number} status on {date}"

# This line starts the server — it listens for JSON-RPC messages via stdio
mcp.run()
'''

print("=== server.py ===")
print(server_code)
print("You'd run this with: python server.py")
print("Then configure Claude Desktop / Cursor to connect to it.")

---

## YOUR TURN: Define Your Own Tool

Create a tool that an AI travel assistant could use. Ideas:
- `search_hotels(city, check_in, check_out)` — returns mock hotel results
- `convert_currency(amount, from_currency, to_currency)` — converts between currencies
- `get_weather(city)` — returns weather for a destination

Requirements:
1. Use the `@mcp.tool()` decorator
2. Add type hints to all parameters
3. Write a clear docstring (this is what the AI sees!)
4. Test it by calling it as a regular function

In [ ]:
# YOUR CODE HERE
# 1. Define a tool with @mcp.tool()
# 2. Add type hints and a docstring
# 3. Implement mock logic (return realistic-looking data)
# 4. Test it below



In [ ]:
# Test your tool here
# Call it as a regular function and print the results



## Key Takeaways

1. **MCP** is an open standard for AI-tool communication — build once, work with any AI client
2. **FastMCP** is the high-level API — like Express.js or Flask for MCP servers
3. **Tools** are actions (`@mcp.tool()`) — things the AI can **do**
4. **Resources** are data (`@mcp.resource()`) — things the AI can **read**
5. **JSON-RPC** is the wire format — discover tools, call them, get results
6. **The logic is just Python** — you can test tools as regular functions

**Next:** In Module 3, you'll build LangChain chains that connect LLMs with prompt templates and output parsers.